# Коломна · 01 · Сбор и проверка данных

Ноутбук воспроизводит **все цифры** из документа «Коломна: исследование локальной идентичности»:

1. границы МО и островов из OpenStreetMap → четыре версии границ Коломны (версии 1–4);
2. здания и точки сервиса из OSM → плотность по зонам;
3. выгрузка ЕГРОКН (JSON из папки проекта) → объекты в Коломне;
4. «Карта возраста домов» (вырезка geopackage) → возраст застройки;
5. население МО (Росстат).

Каждый шаг сохраняет результат в `OUT_DIR` — эти файлы читает ноутбук `02_kolomna_maps.ipynb`.
Запросы к Overpass кэшируются в `OUT_DIR/cache`, поэтому повторный запуск не ходит в сеть.

**Где запускать.** В Colab: смонтируйте Google Drive и положите туда папку `data-69-structure-16.json` и вырезку `hoith_kolomna.gpkg` (см. шаг 4). Локально (Jupyter): укажите путь к папке проекта.

In [ ]:
# Установка (в Colab geopandas и shapely уже есть)
!pip -q install ijson osmnx==2.0.1

In [ ]:
import os, json, re, glob, time, hashlib, collections
import requests, ijson
import pandas as pd, geopandas as gpd
from shapely.geometry import Point, box, shape, mapping
from shapely.ops import unary_union

IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/kolomna-identity'   # ← папка с клоном репозитория
else:
    PROJECT_DIR = os.getcwd()   # корень клона репозитория

EGROKN_DIR = os.path.join(PROJECT_DIR, 'data-69-structure-16.json')
HOITH_CLIP = os.path.join(PROJECT_DIR, 'hoith_kolomna.gpkg')   # вырезка, см. шаг 4
OUT_DIR    = os.path.join(PROJECT_DIR, 'kolomna_data')
CACHE      = os.path.join(OUT_DIR, 'cache')
os.makedirs(CACHE, exist_ok=True)
print(OUT_DIR)

## 0. Overpass с кэшем

Тот же запрос → тот же файл в `cache/`. Дата выгрузки пишется рядом: OSM меняется, и для отчёта важна дата.

In [ ]:
ENDPOINTS = ['https://overpass-api.de/api/interpreter',
             'https://lz4.overpass-api.de/api/interpreter',
             'https://z.overpass-api.de/api/interpreter',
             'https://overpass.kumi.systems/api/interpreter',
             'https://overpass.private.coffee/api/interpreter',
             'https://maps.mail.ru/osm/tools/overpass/api/interpreter']

def overpass(q, tag, rounds=4):
    # Запрос к Overpass с кэшем. Серверы часто перегружены (HTTP 429/504) — перебираем зеркала
    # по кругу, с растущей паузой между кругами. Удачный ответ сохраняется в cache/ и больше не запрашивается.
    fn = os.path.join(CACHE, f'{tag}_{hashlib.md5(q.encode()).hexdigest()[:8]}.json')
    if os.path.exists(fn):
        return json.load(open(fn, encoding='utf-8'))
    log = []
    for rnd in range(rounds):
        for ep in ENDPOINTS:
            try:
                r = requests.post(ep, data={'data': q}, timeout=240, headers={'User-Agent': 'kolomna-itmo-study/1.0'})
                ok = r.status_code == 200 and r.text.lstrip().startswith('{')
                if ok:
                    j = r.json()
                    if j.get('remark') and not j.get('elements'):   # сервер ответил, но запрос оборвался по таймауту
                        log.append(f'{ep}: remark {j["remark"][:80]}'); continue
                    j['_fetched'] = time.strftime('%Y-%m-%d %H:%M'); j['_endpoint'] = ep
                    json.dump(j, open(fn, 'w', encoding='utf-8'), ensure_ascii=False)
                    print(f'{tag}: ok с {ep}')
                    return j
                log.append(f'{ep}: HTTP {r.status_code}')
            except Exception as e:
                log.append(f'{ep}: {type(e).__name__}')
            print('  ', log[-1])
        pause = 30 * (rnd + 1)
        print(f'{tag}: круг {rnd + 1} не удался, жду {pause} с'); time.sleep(pause)
    raise RuntimeError('Overpass недоступен, попробуйте позже. Последние ответы: ' + ' | '.join(log[-6:]))

## 1. Границы: МО и острова из OSM

Используем отношения OSM (id проверяемы на openstreetmap.org/relation/…):

| объект | relation |
|---|---|
| МО Коломна | 1121517 |
| МО Адмиралтейский округ | 1123300 |
| МО Сенной округ | 1121516 |
| МО Екатерингофский | 1123297 |
| Коломенский остров | 3298066 |
| Покровский остров | 3298126 |
| Матисов остров | 1576208 |
| Ново-Адмиралтейский остров | 1576257 |
| остров Новая Голландия | 1576204 |
| Казанский остров | 3298110 |
| Спасский остров | 3298123 |

In [ ]:
REL = {'округ Коломна':1121517, 'Адмиралтейский округ':1123300, 'Сенной округ':1121516,
       'Екатерингофский округ':1123297, 'Коломенский остров':3298066, 'Покровский остров':3298126,
       'Матисов остров':1576208, 'Ново-Адмиралтейский остров':1576257, 'остров Новая Голландия':1576204,
       'Казанский остров':3298110, 'Спасский остров':3298123}
import osmnx as ox
ox.settings.use_cache = True; ox.settings.cache_folder = os.path.join(CACHE, 'osmnx')
polys = ox.geocode_to_gdf([f'R{v}' for v in REL.values()], by_osmid=True)
polys['name_key'] = list(REL.keys())
polys = polys[['name_key','geometry']].set_crs(4326)
polys.to_file(os.path.join(OUT_DIR, 'osm_polygons.geojson'), driver='GeoJSON')
P = {r.name_key: r.geometry.buffer(0) for r in polys.itertuples()}
UTM = 32636
area = lambda g: gpd.GeoSeries([g], crs=4326).to_crs(UTM).area.iloc[0] / 1e6
for k, g in P.items(): print(f'{k:28s} {area(g):6.3f} км²')

## 2. Четыре версии границ Коломны

* **1. «Три острова»** — Коломенский + Покровский + Матисов (краеведы, Википедия, полигон «исторический район Коломна» в Яндекс Картах).
* **2. МО Коломна** — официальная граница (+ Ново-Адмиралтейский остров с верфями, по осям рек).
* **3. «Театральная Коломна»** — версия 2 + Новая Голландия + часть МО Адмиралтейский округ южнее Мойки (Театральная пл., Мариинка, Никольский собор).
* **4. «Литературная Коломна»** — версия 3 + МО Сенной округ + Галерная улица (полоса МО Адмиралтейский округ севернее Мойки западнее 30.3005° в. д.).

Буферы в градусах (0,0003–0,0004° ≈ 20–25 м) только склеивают полигоны островов с полигонами МО через воду.

In [ ]:
mo, adm, sen = P['округ Коломна'], P['Адмиралтейский округ'], P['Сенной округ']
isl = unary_union([P['Коломенский остров'], P['Покровский остров'], P['Матисов остров']])
nh = P['остров Новая Голландия']
south = unary_union([P['Казанский остров'], P['Спасский остров']]).buffer(0.0004)
adm_s = adm.intersection(south)
adm_n = adm.difference(south).difference(nh.buffer(0.0003))
galer = adm_n.intersection(box(30.27, 59.925, 30.3005, 59.95))
VARIANTS = {
  'версия 1': ('«Три острова» (краеведческая)', isl),
  'версия 2': ('МО Коломна (официальная)', mo),
  'версия 3': ('«Театральная Коломна» (расширенная)', unary_union([mo, nh.buffer(0.0003), adm_s]).buffer(0)),
  'версия 4': ('«Литературная Коломна» (максимальная)', unary_union([mo, nh.buffer(0.0003), adm_s, sen, galer]).buffer(0)),
}
var = gpd.GeoDataFrame([{'key':k, 'label':l, 'area_km2':round(area(g),2), 'geometry':g} for k,(l,g) in VARIANTS.items()], crs=4326)
var.to_file(os.path.join(OUT_DIR, 'kolomna_boundary_variants.geojson'), driver='GeoJSON')
var[['key','label','area_km2']]

## 3. Здания и сервис из OSM

Запросы ровно те, что использованы в документе (`out tags center` — точка-центр для каждого объекта). Каждый объект относим к версиям границ по его центру.

In [ ]:
BBOX = '59.913,30.258,59.939,30.33'
Q_BLD = f'[out:json][timeout:180];(way["building"]({BBOX});relation["building"]({BBOX}););out tags center;'
Q_POI = f'''[out:json][timeout:180];(node["amenity"]({BBOX});node["shop"]({BBOX});node["tourism"]({BBOX});
node["leisure"]({BBOX});way["amenity"]({BBOX});way["leisure"]({BBOX});way["tourism"]({BBOX});
node["office"]({BBOX});node["craft"]({BBOX}););out tags center;'''
bld_j = overpass(Q_BLD, 'buildings')
poi_j = overpass(Q_POI, 'poi')
print('здания', len(bld_j['elements']), bld_j['_fetched'], '| точки', len(poi_j['elements']), poi_j['_fetched'])

def to_gdf(j):
    rows = []
    for e in j['elements']:
        c = e.get('center') or ({'lat':e['lat'],'lon':e['lon']} if 'lat' in e else None)
        if not c: continue
        rows.append({'osm':f"{e['type']}/{e['id']}", **{k:e['tags'].get(k) for k in
            ['building','building:levels','start_date','amenity','shop','tourism','leisure','office','craft','name']},
            'geometry': Point(c['lon'], c['lat'])})
    return gpd.GeoDataFrame(rows, crs=4326)
bld, poi = to_gdf(bld_j), to_gdf(poi_j)
for k,(l,g) in VARIANTS.items():
    bld[k] = bld.within(g); poi[k] = poi.within(g)

def cat(r):
    if r.amenity: return 'a:'+r.amenity
    if r.shop: return 'shop'
    if r.tourism: return 't:'+r.tourism
    if r.leisure: return 'l:'+r.leisure
    if r.office: return 'office'
    if r.craft: return 'craft'
poi['cat'] = poi.apply(cat, axis=1)
FOOD = {'a:cafe','a:restaurant','a:bar','a:fast_food'}
SERV = FOOD | {'shop','t:hotel','t:hostel'}
poi['food'] = poi.cat.isin(FOOD); poi['serv'] = poi.cat.isin(SERV)
bld.to_file(os.path.join(OUT_DIR,'osm_buildings_points.geojson'), driver='GeoJSON')
poi.to_file(os.path.join(OUT_DIR,'osm_poi.geojson'), driver='GeoJSON')

In [ ]:
# Сводка по версиям границ
lv = pd.to_numeric(bld['building:levels'], errors='coerce')
summary = []
for k,(l,g) in VARIANTS.items():
    s = poi[poi[k]]
    summary.append({'версия':k, 'км²':round(area(g),2), 'зданий OSM':int(bld[k].sum()),
        'медиана этажей':lv[bld[k]].median(), 'с датой постройки':int(bld[k][bld.start_date.notna()].sum()),
        'магазины':int((s.cat=='shop').sum()), 'общепит':int(s.food.sum()), 'гостиницы+хостелы':int(s.cat.isin(['t:hotel','t:hostel']).sum()),
        'аптеки':int((s.cat=='a:pharmacy').sum()), 'детсады':int((s.cat=='a:kindergarten').sum()), 'школы':int((s.cat=='a:school').sum())})
summary = pd.DataFrame(summary); summary

In [ ]:
# Градиент «ядро → кольца» (таблица в разделе «Границы»)
def ring(a, b=None):
    ga = VARIANTS[a][1]; g = ga if b is None else ga.difference(VARIANTS[b][1])
    m = poi.within(g); A = area(g)
    return A, poi[m].serv.sum()/A, poi[m].food.sum()/A
rows = []
for name,a,b in [('Ядро (версия 1)','версия 1',None), ('Кольцо «театральное» (версия 3 без 2)','версия 3','версия 2'), ('Кольцо «литературное» (версия 4 без 3)','версия 4','версия 3')]:
    A, s, f = ring(a,b); rows.append({'зона':name,'км²':round(A,2),'сервис на км²':round(s),'общепит на км²':round(f)})
gradient = pd.DataFrame(rows); gradient

## 4. ЕГРОКН: объекты в Коломне

Читаем 17 JSON потоково (`ijson`, ~1,2 ГБ), оставляем Санкт-Петербург (region.id = 78). Объекты с точкой относим к версиям границ; три объекта без координат добавляем по адресу (они указаны явно ниже — проверьте).

In [ ]:
spb = []
files = sorted(glob.glob(os.path.join(EGROKN_DIR, '*.json')), key=lambda p: int(re.findall(r'(\d+)\.json$', p)[0]))
total = 0
for p in files:
    with open(p, 'rb') as f:
        for o in ijson.items(f, 'item', use_float=True):
            total += 1
            g = o.get('data', {}).get('general', {})
            a = g.get('address') or {}
            if (g.get('region') or {}).get('id') != '78' and 'Санкт-Петербург' not in (a.get('fullAddress') or ''):
                continue
            mp = a.get('mapPosition') or {}
            c = mp.get('coordinates') if mp.get('type') == 'Point' else None
            spb.append({'reg':g.get('regNumber'), 'name':g.get('name'), 'type':(g.get('objectType') or {}).get('value'),
                        'cat':(g.get('categoryType') or {}).get('value'),
                        'typology':'; '.join(t['value'] for t in g.get('typologies') or []),
                        'date':g.get('createDate'), 'address':a.get('fullAddress'),
                        'lon':c[0] if c else None, 'lat':c[1] if c else None})
spb = pd.DataFrame(spb)
for c in ['reg', 'name', 'typology', 'date', 'address']:   # в выгрузке встречаются лишние пробелы и табуляции
    if c in spb: spb[c] = spb[c].astype('string').str.strip()
if files:
    print('всего записей', total, '| СПб', len(spb), '| СПб без координат', spb.lon.isna().sum())
else:   # JSON ЕГРОКН не загружены — берём готовую выборку по Коломне, собранную этой же ячейкой из полной выгрузки
    READY = os.path.join(PROJECT_DIR, 'kolomna_data', 'okn_kolomna_egrokn.csv')
    spb = pd.read_csv(READY)[['reg','name','type','cat','typology','date','address','lon','lat']]
    spb['reg'] = spb['reg'].astype(str).str.strip()
    print('JSON ЕГРОКН не найдены в', EGROKN_DIR, '— использую готовую выборку', READY, len(spb), 'записей')

In [ ]:
okn = gpd.GeoDataFrame(spb, geometry=[Point(x,y) if pd.notna(x) else None for x,y in zip(spb.lon, spb.lat)], crs=4326)
# здания — строго внутри полигона; мосты, набережные, каналы — с допуском 30 м: границы версий идут по воде
# (Фонтанка, Мойка, Крюков канал, Пряжка), и пограничные мосты — часть Коломны (семимостье, Египетский мост)
LINEAR = r'\bмост|набережн|\bканал\b|спуск'
_pm = okn.geometry.to_crs(32636)
_vm = {k: gpd.GeoSeries([g], crs=4326).to_crs(32636)[0] for k, (l, g) in VARIANTS.items()}
_lin = okn.name.fillna('').str.lower().str.contains(LINEAR)
for k in VARIANTS:
    okn[k] = [bool(p) and (_vm[k].buffer(30).contains(q) if lin else _vm[k].contains(q)) for p, q, lin in zip(okn.geometry, _pm, _lin)]
# но мосты на внешнем краю версии 4 в Сенном округе (Каменный, Обуховский, Измайловский) — там край версии
# проходит по границе МО Сенной округ, а не Коломны; мосты на Неве и на исторических границах Коломны остаются
_sen = gpd.GeoSeries([sen], crs=4326).to_crs(32636)[0]
_edge = [bool(p and re.search(r'\bмост', str(n).lower()) and _vm['версия 4'].buffer(30).contains(q)
              and not _vm['версия 3'].buffer(30).contains(q) and _vm['версия 4'].boundary.distance(q) < 30
              and _sen.buffer(30).contains(q))
         for n, p, q in zip(okn.name, okn.geometry, _pm)]
okn.loc[_edge, 'версия 4'] = False
ADDR_IN_MO = ['улица Союза Печатников, дом 5', 'проспект Римского-Корсакова, дом 37', 'площадь Репина, дом 3-5']
okn['in_mo_by_address'] = okn.geometry.isna() & okn.address.fillna('').apply(lambda s: any(a in s for a in ADDR_IN_MO))
print(okn[okn.in_mo_by_address][['name','address']].to_string())
okn['year'] = okn.date.fillna('').apply(lambda s: min(map(int, re.findall(r'(1[6-9]\d\d|20[0-2]\d)', s)), default=None))
sel = okn[okn['версия 4'] | okn.in_mo_by_address].copy()
sel.drop(columns='geometry').to_csv(os.path.join(OUT_DIR,'okn_kolomna_egrokn.csv'), index=False)
sel[sel.geometry.notna()].to_file(os.path.join(OUT_DIR,'okn_kolomna_egrokn.geojson'), driver='GeoJSON')
pd.DataFrame({k:[int(okn[k].sum()) + (int(okn.in_mo_by_address.sum()) if k=='версия 2' else 0)] for k in VARIANTS}, index=['ОКН'])

In [ ]:
# Категории ОКН внутри МО (версия 2 + по адресу)
m = okn['версия 2'] | okn.in_mo_by_address
okn[m].cat.value_counts()

## 5. «Карта возраста домов» (HOITH)

Файл `HOITH_database_231118.gpkg` весит 1,8 ГБ — вырежьте Коломну у себя одной командой (нужен GDAL: `sudo apt install gdal-bin`):

```bash
cd ~/Claude/"Городская идентичность"
ogrinfo -so HOITH_database_231118.gpkg            # посмотреть имя слоя
ogr2ogr -f GPKG hoith_kolomna.gpkg HOITH_database_231118.gpkg \
        -spat 30.255 59.910 30.335 59.942 -spat_srs EPSG:4326
```

Получится файл на несколько МБ. Положите его рядом с ноутбуком (`HOITH_CLIP`).

In [ ]:
if os.path.exists(HOITH_CLIP):
    import pyogrio
    layers = [l for l, _ in pyogrio.list_layers(HOITH_CLIP)]; print('слои:', layers)   # all_cities_181123, EPSG:3857
    h = gpd.read_file(HOITH_CLIP, layer=layers[0]).to_crs(4326)
    print(h.columns.tolist()); display(h.head())
else:
    print('Нет вырезки — выполните команду ogr2ogr выше')

In [ ]:
YEAR_COL = 'r_year_int'   # год постройки в выгрузке HOITH 2023-11-18
if os.path.exists(HOITH_CLIP) and YEAR_COL:
    h['year'] = pd.to_numeric(h[YEAR_COL], errors='coerce')
    bins = [0,1800,1850,1880,1918,1956,1992,2100]
    labels = ['до 1800','1800–1849','1850–1879','1880–1917','1918–1955','1956–1991','1992+']
    h['period'] = pd.cut(h.year, bins=bins, labels=labels, right=False)
    for k,(l,g) in VARIANTS.items():
        h[k] = h.representative_point().within(g)
    h['area_m2'] = h.to_crs(UTM).area
    for k in VARIANTS:
        s = h[h[k]]
        print(f"{k}: зданий {len(s)}, с годом {s.year.notna().sum()} ({s.year.notna().mean():.0%}), медиана {s.year.median():.0f}, до 1918 — {(s.year<1918).sum()/s.year.notna().sum():.1%}")
    ring = h[h['версия 4'] & ~h['версия 2']]
    shares = pd.DataFrame({
        'МО Коломна (версия 2)': h[h['версия 2']].groupby('period', observed=False).area_m2.sum(),
        'восточные кольца (версия 4 без 2)': ring.groupby('period', observed=False).area_m2.sum()})
    display((shares / shares.sum() * 100).round(1))   # график «Ядро моложе окраин» в документе
    print('медиана года в кольцах:', ring.year.median())
    display(h[h['версия 2']].r_style.value_counts().head(8))
    out = h[h['версия 4']].copy(); out['period'] = out.period.astype(str)
    out.to_file(os.path.join(OUT_DIR,'hoith_kolomna_periods.gpkg'), driver='GPKG')

## 6. Население МО Коломна (Петростат)

Источник — PDF Петростата из папки проекта (https://78.rosstat.gov.ru/folder/27595): оценки численности по МО на 1 января 2018–2025 гг. и экспресс-итоги ВПН-2020 (с графой ВПН-2010). Таблицу собирает скрипт `kolomna/scripts/population_from_pdfs.py` (нужен `pdftotext`).

**Внимание:** оценки 2018–2022 гг. считались от переписи 2010 г., с 2023 г. — от переписи 2020 г. (октябрь 2021). Скачок 2022→2023 — пересчёт базы. Сравнивайте перепись с переписью.

In [ ]:
import sys, shutil, subprocess; sys.path.insert(0, os.path.join(PROJECT_DIR, 'scripts'))
from population_from_pdfs import build, FILES
SRC_DIR = PROJECT_DIR if os.path.exists(os.path.join(PROJECT_DIR, FILES['01.01.2025'])) else os.path.join(PROJECT_DIR, 'sources_petrostat')
HAVE_PDF = os.path.exists(os.path.join(SRC_DIR, FILES['01.01.2025']))
if HAVE_PDF and not shutil.which('pdftotext'):   # pdftotext нужен только для пересборки из PDF
    subprocess.run('apt-get -qq update && apt-get -qq install -y poppler-utils', shell=True, check=False)
if HAVE_PDF:
    pop = build(SRC_DIR)          # пересобираем из PDF Петростата
else:                             # PDF нет — берём готовую таблицу, собранную тем же скриптом
    pop = pd.read_csv(os.path.join(PROJECT_DIR, 'kolomna_data', 'population_spb_mo_2010_2025.csv'), index_col=0)
pop.to_csv(os.path.join(OUT_DIR, 'population_spb_mo_2010_2025.csv'), encoding='utf-8-sig')
adm = pop[(pop['район'] == 'Адмиралтейский район') | pop.index.isin(['Санкт-Петербург', 'Адмиралтейский район'])]
display(adm)
mo = pop[pop.index.str.startswith('МО ')].sort_values('ВПН-2020 к ВПН-2010, %')
rank = list(mo.index).index('МО Коломна') + 1
print(f'Коломна: ВПН-2010 → ВПН-2020 = {mo.loc["МО Коломна", "ВПН-2020 к ВПН-2010, %"]} %, '
      f'{rank}-е место по убыли из {mo["ВПН-2020 к ВПН-2010, %"].notna().sum()} МО; город — {pop.loc["Санкт-Петербург", "ВПН-2020 к ВПН-2010, %"]} %')
mo[['район', 'ВПН-2020 к ВПН-2010, %']].head(10)

In [ ]:
# Возрастно-половой состав по районам на 01.01.2024 (бюллетень Петростата 14001024; по МО данных нет)
from age_sex_from_pdf import build as build_age, FILE as AGE_FILE
age_pdf = os.path.join(SRC_DIR, AGE_FILE)
if os.path.exists(age_pdf):
    age = build_age(age_pdf)
else:
    age = pd.read_csv(os.path.join(PROJECT_DIR, 'kolomna_data', 'age_sex_spb_districts_2024.csv'), index_col=0)
age.to_csv(os.path.join(OUT_DIR, 'age_sex_spb_districts_2024.csv'), encoding='utf-8-sig')
age.loc[['Санкт-Петербург', 'Адмиралтейский район', 'Центральный район', 'Петроградский район']].T

In [ ]:
# Всё, что сохранено
sorted(os.listdir(OUT_DIR))